In [1]:
import numpy as np
import math
import csv
import matplotlib.pyplot as plt
from collections import Counter
from functools import lru_cache
# ==========================================
# MATHEMATICAL CORE (OPTIMIZED)
# ==========================================

from fractions import Fraction


import numpy as np
from pathlib import Path
from math import gcd

CACHE_PATH = Path("dedekind_cache.npz")

def _ds(p: int, q: int) -> float:
    p = p % q
    result, sign = 0.0, 1
    while p > 1:
        result += sign * (p*p + q*q + 1 - 3.0*p*q) / (12.0*p*q)
        sign = -sign
        p, q = q % p, p
    if p == 1:
        result += sign * (q - 1) * (q - 2) / (12.0 * q)
    return result

def build_cache(q_max: int):
    """Precompute all s(p, q) for q <= q_max, save to disk."""
    # store as flat arrays: keys (p,q) packed into int64, values as float64
    keys = []
    vals = []
    for q in range(2, q_max + 1):
        for p in range(1, q):
            if gcd(p, q) == 1:
                keys.append((p << 32) | q)  # pack p,q into one int64
                vals.append(_ds(p, q))
    np.savez_compressed(CACHE_PATH,
                        keys=np.array(keys, dtype=np.int64),
                        vals=np.array(vals, dtype=np.float64))
    print(f"Cached {len(keys)} values to {CACHE_PATH}")

def load_cache() -> dict[int, float]:
    """Load cache into a dict for O(1) lookup."""
    data = np.load(CACHE_PATH)
    return dict(zip(data["keys"].tolist(), data["vals"].tolist()))

# --- usage ---
# once:
build_cache(q_max=5000)

# every subsequent run:
_cache = load_cache()

def fast_dedekind_sum(p: int, q: int) -> float:
    p = p % q
    if p == 0 or q <= 1:
        return 0.0
    key = (p << 32) | q
    try:
        return _cache[key]
    except KeyError:
        # fallback: compute on the fly (non-coprime, or outside q_max)
        return _ds(p, q)


def compute_K2_plus_s(e0: int, arms: list) -> float:
    """
    Computes K^2 + s using Equation (2.4) from Borodzik-Nemethi:
    K^2 + s = epsilon^2 * e + e + 5 - 12 * sum(s(omega_l, alpha_l))
    """
    nu = len(arms)
    
    # Calculate e = e_0 + sum(omega_l / alpha_l)
    e = e0 + sum(w / a for w, a in arms)
    
    # Calculate epsilon = (2 - nu + sum(1 / alpha_l)) / e
    epsilon = (2 - nu + sum(1.0 / a for w, a in arms)) / e
    
    # Calculate the sum of the classical Dedekind sums for all arms in O(log(alpha_l))
    dedekind_total = sum(fast_dedekind_sum(w, a) for w, a in arms)
    
    # Assemble Eq (2.4)
    K2_plus_s = (epsilon**2) * e + e + 5 - 12 * dedekind_total
    
    return K2_plus_s

def brieskorn_seifert(p: int, q: int, r: int):
    """
    Seifert invariants for Sigma(p, q, r).
    Normalized: omega_i = (-alpha_hat_i^{-1}) mod alpha_i, so 0 < omega_i < alpha_i.
    """
    alpha = [p, q, r]
    alpha_hat = [q * r, p * r, p * q]

    omega = []
    for i in range(3):
        inv = pow(alpha_hat[i], -1, alpha[i])
        omega.append((-inv) % alpha[i])

    alpha_prod = p * q * r
    rhs = -1 - sum(omega[i] * alpha_hat[i] for i in range(3))
    assert rhs % alpha_prod == 0, f"e0 not integer for Sigma({p},{q},{r})"
    e0 = rhs // alpha_prod

    return e0, list(zip(omega, alpha))

def compute_tau(e0: int, arms: list, m_max: int):
    """
    Vectorized computation of tau values using NumPy.
    Runs in O(m_max) vectorized C-time instead of iterating a Python loop.
    """
    j = np.arange(m_max)
    
    # Calculate delta_j array for all j simultaneously
    delta_j = 1 - j * e0
    for w, a in arms:
        # (x + a - 1) // a is mathematically identical to ceiling division -(-x // a)
        # but executes cleanly in vectorized numpy without floating point precision issues.
        delta_j -= (j * w + a - 1) // a 
        
    # tau[0] = 0, subsequent elements are the cumulative sum of delta_j
    tau = np.zeros(m_max + 1, dtype=int)
    np.cumsum(delta_j, out=tau[1:])
    
    return tau

def d_invariant_exact(p: int, q: int, r: int):
    """
    d(Sigma(p,q,r)) = (K^2+s)/4 - 2*min_{m>=0} tau(m)
    Returns (d_invariant, K^2+s/4, 2*min_tau)
    """
    # Calculate the Seifert invariants directly 
    e0, arms = brieskorn_seifert(p, q, r)
    
    # Compute K^2 + s using the new O(1) mathematical formulation
    ks = compute_K2_plus_s(e0, arms)

    # Compute tau values and find the minimum
    tau_vals = compute_tau(e0, arms, p * q * r)
    min_tau = np.min(tau_vals) # Using numpy's min over the generated array
    
    ks_term = int(np.round(ks / 4.0))
    tau_term = 2 * int(np.round(min_tau))
    d = ks_term - tau_term

    return d, ks_term, tau_term

Cached 7600457 values to dedekind_cache.npz


In [2]:
import math

# ==========================================
# THEOREM N CROSS-REFERENCING
# ==========================================

def is_theorem_n_known(p, q, r):
    """
    Checks if Sigma(p, q, r) falls into the explicit examples or 
    infinite families described in Theorem N, or custom known families.
    """
    # Explicit checks
    if (p, q, r) in [(2, 3, 13), (2, 3, 25), (2, 7, 19), (3, 5, 19), (2, 7, 47), (3, 5, 49)]:
        return True

    # Family 1: p even, s odd, q = ps-1, r = ps+1 (or vice versa)
    if p % 2 == 0:
        for s in range(1, (r + 2) // p + 2, 2):
            if {q, r} == {p*s - 1, p*s + 1}:
                return True

    # Family 2: p odd, s arbitrary, {q, r} == {ps ± 1, ps ± 2} 
    if p % 2 != 0:
        for s in range(1, (r + 3) // p + 2):
            for e1 in [-1, 1, -2, 2]:
                for e2 in [-1, 1, -2, 2]:
                    if abs(e1) != abs(e2):
                        if {q, r} == {p*s + e1, p*s + e2}:
                            return True

    # Family 3: p=2, q=2s ± 1, r=4(2s ± 1) + 2s ∓ 1 for s odd
    if p == 2:
        for s in range(1, q + 2, 2):
            for sign in [-1, 1]:
                test_q = 2*s + sign
                test_r = 4*test_q + 2*s - sign
                if {q, r} == {test_q, test_r}:
                    return True

    # Family 4: p=3, q=3s ± 1, r=6(3s ± 1) + 3s ∓ 2 for s arbitrary
    if p == 3:
        for s in range(1, q + 2):
            for sign in [-1, 1]:
                test_q = 3*s + sign
                test_r = 6*test_q + 3*s - 2*sign
                if {q, r} == {test_q, test_r}:
                    return True

    # Family 5: p=3, q=3s ± 2, r=6(3s ± 2) + 3s ∓ 1 for s arbitrary
    if p == 3:
        for s in range(1, q + 2):
            for sign in [-1, 1]:
                test_q = 3*s + 2*sign
                test_r = 6*test_q + 3*s - sign
                if {q, r} == {test_q, test_r}:
                    return True

    # NEW FAMILY: p, q coprime, r = pqn + 1 for n >= 1
    # Note: This automatically covers the (2, 5, 10n + 1) and (2, 7, 14n + 1) families
    if math.gcd(p, q) == 1:
        if (r - 1) % (p * q) == 0 and (r - 1) // (p * q) >= 1:
            return True

    # NEW FAMILY: (2, 5, 10n - 3) for n >= 1
    if p == 2 and q == 5:
        if (r + 3) % 10 == 0 and (r + 3) // 10 >= 1:
            return True
            
    # NEW FAMILY: (2, 7, 14n + 3) for n >= 1
    if p == 2 and q == 7:
        if (r - 3) % 14 == 0 and (r - 3) // 14 >= 1:
            return True
            
    # NEW FAMILY: (2, 7, 14n + 5) for n >= 1
    if p == 2 and q == 7:
        if (r - 5) % 14 == 0 and (r - 5) // 14 >= 1:
            return True

    return False

$\Sigma(2, 11, 8n-1)$ for $n \geq 2$.

In [3]:
import math

# You can adjust the upper bound for n here
MAX_N = 30  

print("Calculating d-invariants for Family: Sigma(2, 11, 8n-1)")
print("Constraints: n >= 2\n")
print(f"{'p':>3} | {'q':>3} | {'n':>3} | {'r':>5} | {'d-invariant':>12} | {'K^2+s/4':>10} | {'2*min_tau':>10}")
print("-" * 66)

p = 2
q = 11

for n in range(2, MAX_N + 1):
    r = 8 * n - 1
    
    # Check coprimality condition. (r is always odd so gcd(2, r) = 1, 
    # but we must ensure gcd(11, r) == 1)
    if math.gcd(p, r) == 1 and math.gcd(q, r) == 1:
        try:
            d, ks_term, tau_term = d_invariant_exact(p, q, r)
            print(f"{p:3} | {q:3} | {n:3} | {r:5} | {round(d):12} | {round(ks_term):10} | {round(tau_term):10}")
        except Exception as e:
            print(f"Error computing Sigma({p},{q},{r}): {e}")

Calculating d-invariants for Family: Sigma(2, 11, 8n-1)
Constraints: n >= 2

  p |   q |   n |     r |  d-invariant |    K^2+s/4 |  2*min_tau
------------------------------------------------------------------
  2 |  11 |   2 |    15 |            0 |        -10 |        -10
  2 |  11 |   3 |    23 |            0 |        -20 |        -20
  2 |  11 |   4 |    31 |            0 |        -24 |        -24
  2 |  11 |   5 |    39 |            4 |        -28 |        -32
  2 |  11 |   6 |    47 |            2 |        -38 |        -40
  2 |  11 |   8 |    63 |            2 |        -52 |        -54
  2 |  11 |   9 |    71 |            0 |        -62 |        -62
  2 |  11 |  10 |    79 |            2 |        -66 |        -68
  2 |  11 |  11 |    87 |            6 |        -70 |        -76
  2 |  11 |  12 |    95 |            2 |        -80 |        -82
  2 |  11 |  13 |   103 |            0 |        -90 |        -90
  2 |  11 |  14 |   111 |            0 |       -100 |       -100
  2 |  11 |

In [ ]:

# ==========================================
# PARAMETER EXECUTION AND CSV EXPORT
# ==========================================

N = 300
results = []

print(f"Scanning pairwise coprime triplets (p, q, r) for p < q < r < {N}...")

for p in range(2, N):
    for q in range(p + 1, N):
        if math.gcd(p, q) != 1:
            continue
        for r in range(q + 1, N):
            if math.gcd(p, r) == 1 and math.gcd(q, r) == 1:
                try:
                    # Unpack the new tuple from d_invariant_exact
                    d, ks_term, tau_term = d_invariant_exact(p, q, r)
                    
                    # Update status checking: if d isn't 0, it's N/A for Theorem N
                    if round(d) != 0:
                        status = "N/A"
                    else:
                        status = "Known" if is_theorem_n_known(p, q, r) else "Unknown"
                        
                    results.append({
                        'p': p, 
                        'q': q, 
                        'r': r, 
                        'd_invariant': round(d),
                        'K^2+s/4': round(ks_term),
                        '2*min_tau': round(tau_term),
                        'Status': status
                    })
                except Exception as e:
                    print(f"Error computing Sigma({p},{q},{r}): {e}")

csv_filename = 'brieskorn_d_all_300.csv'
with open(csv_filename, 'w', newline='') as csvfile:
    fieldnames = ['p', 'q', 'r', 'd_invariant', 'K^2+s/4', '2*min_tau', 'Status']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for row in results:
        writer.writerow(row)

print(f"\nSearch complete. Computed {len(results)} spheres in total.")
print(f"Data successfully exported to {csv_filename}")

# ==========================================
# VISUALIZATION
# ==========================================

# Extract rounded d-invariants to handle any floating point imprecision
d_values = [round(row['d_invariant']) for row in results]
counts = Counter(d_values)
total_spheres = len(d_values)

# Sort the unique d values for an ordered x-axis
sorted_d = sorted(counts.keys())
frequencies = [counts[d] for d in sorted_d]
percentages = [(f / total_spheres) * 100 for f in frequencies]

# Generate the Bar Chart
plt.figure(figsize=(14, 7))
bars = plt.bar([str(d) for d in sorted_d], frequencies, color='steelblue', edgecolor='black')

plt.xlabel('d-invariant', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title(f'Distribution of d-invariants for Brieskorn Spheres (N={N})', fontsize=14)

# Add percentage labels directly on top of each bar
for bar, pct in zip(bars, percentages):
    yval = bar.get_height()
    # Offset the text slightly above the bar based on the max frequency height
    plt.text(bar.get_x() + bar.get_width()/2, yval + (max(frequencies) * 0.015), 
             f'{pct:.1f}%', ha='center', va='bottom', fontsize=9, rotation=45)

plt.margins(y=0.15) # Leave room at the top for the labels
plt.tight_layout()
plt.show()

Scanning pairwise coprime triplets (p, q, r) for p < q < r < 300...


In [ ]:
counts

In [271]:
_phi = list(range(10000 + 1))
for p in range(2, 10000 + 1):
    if _phi[p] == p:
        for m in range(p, 10000 + 1, p):
            _phi[m] -= _phi[m] // p

def euler_totient(n: int) -> int:
    return _phi[n]

In [231]:
def third_parameter_gives_d_0(p,q,N):
    candidates = []
    if math.gcd(p,q) != 1:
        raise Exception("p and q must be coprime")
    for r in range(N):
        if math.gcd(p,r) == 1 and math.gcd(q,r) == 1:
            if d_invariant_exact(p,q,r)[0]==4:
                candidates.append(r)
    return candidates

def third_parameter_residues_classes(p,q,N):
    res_classes = []
    if math.gcd(p,q) != 1:
        raise Exception("p and q must be coprime")
    return list(set([a % (p*q) for a in third_parameter_gives_d_0(p,q,N)]))

def second_and_third_parameter_gives_d_0(p,N):
    candidates = []
    for q in range(p+1,N):
        if math.gcd(p,q) == 1:
            for r in range(q+1,N):
                if math.gcd(q,r) == 1 and math.gcd(p,r) == 1:
                    if d_invariant_exact(p,q,r)[0]==0:
                        candidates.append((q,r))
    return candidates


In [7]:
results

NameError: name 'results' is not defined

In [46]:
def crt_invert(a, b, c):
    """
    CRT-corrected inversion in the b-component.
    Given pairwise coprime (a, b, c), returns (a', b, c) where
        a' ≡ a^{-1} (mod b)
        a' ≡ a      (mod c)
    and 0 <= a' < b*c. The result is pairwise coprime.
    """
    assert math.gcd(a, b) == math.gcd(b, c) == math.gcd(a, c) == 1, "must be pairwise coprime"

    a_inv_mod_b = pow(a, -1, b)          # Python 3.8+: modular inverse
    a_mod_c     = a % c

    # CRT: find a' with a' ≡ a_inv_mod_b (mod b), a' ≡ a_mod_c (mod c)
    # Since gcd(b, c) = 1, the solution mod b*c is unique.
    b_inv_mod_c = pow(b, -1, c)
    k = ((a_mod_c - a_inv_mod_b) * b_inv_mod_c) % c
    a_prime = a_inv_mod_b + b * k        # in [0, b*c)

    return (a_prime, b, c)

In [50]:
with open('brieskorn_d_all_200.csv', newline='') as f:
    reader = csv.reader(f)
    header = next(reader)
    rows = list(reader)

for row in rows:
    p, q, r, d = int(row[0]), int(row[1]), int(row[2]), int(row[3])
    if d == 0:
        inv = crt_invert(p,q,r)
        print(d_invariant_exact(*inv))
    

(0, 0, 0)
(0, 0, 0)
(0, 0, 0)
(0, 0, 0)
(0, 0, 0)
(0, 0, 0)
(0, 0, 0)
(0, 0, 0)
(0, 0, 0)
(0, 0, 0)
(0, 0, 0)
(0, 0, 0)
(0, 0, 0)
(0, 0, 0)
(0, 0, 0)
(0, 0, 0)
(0, 0, 0)
(0, 0, 0)
(0, 0, 0)
(4, -72, -76)
(0, -72, -72)
(4, -584, -588)
(0, -304, -304)
(4, -1576, -1580)
(0, -696, -696)
(4, -3048, -3052)
(0, -1248, -1248)
(4, -5000, -5004)
(0, -1960, -1960)
(4, -7432, -7436)
(0, -2832, -2832)
(4, -10344, -10348)
(0, -3864, -3864)
(4, -13736, -13740)
(0, -5056, -5056)
(4, -17608, -17612)
(0, -6408, -6408)
(4, -21960, -21964)
(0, -7920, -7920)
(4, -26792, -26796)
(0, -9592, -9592)
(4, -32104, -32108)
(0, -486, -486)
(0, -960, -960)
(2, -2442, -2444)
(0, -1980, -1980)
(0, -3420, -3420)
(2, -7800, -7802)
(0, -4482, -4482)
(0, -7392, -7392)
(2, -16182, -16184)
(0, -7992, -7992)
(0, -12876, -12876)
(2, -27588, -27590)
(0, -12510, -12510)
(0, -19872, -19872)
(2, -42018, -42020)
(0, -18036, -18036)
(0, -28380, -28380)
(2, -59472, -59474)
(0, -24570, -24570)
(0, -38400, -38400)
(2, -79950, -79952)


KeyboardInterrupt: 